In [8]:
import xarray as xr
import os, glob
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm

In [4]:
city_name = 'Moscow'
city_lon = 37.62
city_lat = 55.75

# city_name = 'Tomsk'
# city_lon = 84.96
# city_lat = 56.46

# city_name = 'Beijing'
# city_lon = 116.37
# city_lat = 39.97


ERA5_dir = r'D:\!Work\IFA\!data\Urban_fluxes_NE\ERA5\%s'%city_name

rea_data = xr.open_mfdataset (ERA5_dir + '/*.nc')

rea_df = rea_data.sel (latitude = city_lat, longitude = city_lon, method = 'nearest').to_dataframe()

rea_df['t2m'] = rea_df['t2m'] - 273.15
rea_df['d2m'] = rea_df['d2m'] - 273.15
rea_df['sp'] = rea_df['sp']/100

rea_df['rho'] = (rea_df['sp']*100) / (286.7 * (rea_df['t2m'] + 273.15))

rea_df['rh2m'] = 100 * np.exp((17.27 * (rea_df['d2m'] - rea_df['t2m']) / (237.3 + rea_df['d2m'])))


rea_df['sshf'] = - rea_df['sshf'] / 3600 
rea_df['slhf'] = - rea_df['slhf'] / 3600 
rea_df['vel10'] = np.sqrt (rea_df['u10']**2 + rea_df['v10']**2)
rea_df['tau'] = 1.2 * rea_df['zust']**2


%matplotlib qt
rea_df['rho'].plot()



<Axes: xlabel='valid_time'>

In [7]:
export_dir = "D:/!Work/IFA/!data/Urban_fluxes_NE/processed/%s"%city_name
os.makedirs(export_dir, exist_ok=True)

df2export = rea_df[['sshf', 'slhf', 'tau', 'zust', 't2m', 'rh2m', 'sp', 'rho', 'u10', 'v10', 'vel10']]
df2export = df2export.rename(columns = {'sshf': 'Qh',
                                        'slhf': 'Qle',
                                        'tau': 'Qtau',
                                        'zust':'Ustar', 
                                        't2m': 'T_air',
                                        'rh2m': 'RH_air',
                                        'sp': 'P_air',
                                        'rho': 'Rho_air',
                                        'v10':'Wind_V',
                                        'u10':'Wind_U',
                                        'w':'Wind_W',
                                        'vel10':'Wind_vel'})
df2export.index = df2export.index
df2export.to_csv(export_dir + '/' + 'ERA5.csv')

In [ ]:
rea_data_cr = rea_data.sel(valid_time=slice(pd.Timestamp(2024,6,1),pd.Timestamp(2024,7,1)))

plt.figure()
(-rea_data_cr['slhf']/3600).mean(dim='valid_time').plot()
plt.plot (city_lon, city_lat, 'ok')